# Retrain the 5-model ensemble — Colab

Written for the **rebuilt dataset** (128,400 windows, all six SNR bins). The
previous `data/processed` on the work machine held only 642 windows across two
SNR bins (`0` and `10`), neither of which matches `configs/default.yaml`'s
`snr_bins_db: [-10, -6, -2, 2, 6, 10]` — so nothing trained or evaluated on it
could reproduce the accuracy-vs-SNR sweep the project reports. That is what
this run replaces.

**What comes out of this notebook:** `results/ensemble_0..4.pt` and
`results/best_model.pt`. Download them, then finish **locally** —
threshold calibration is not run here, on purpose (see the last section).

**Runtime → Change runtime type → GPU** before starting, or this takes hours
instead of tens of minutes.

## 1. Get the code

Colab clones from GitHub, so whatever branch you name here must be **pushed**
first. The training pipeline (`src/train.py`, `scripts/train_ensemble.py`) is
identical on `main` and `onnx-export` — the ONNX/static-site work only added
files — so either works. `main` is the safe default.

In [ ]:
BRANCH = 'main'

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b $BRANCH https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log --oneline -1

In [ ]:
# Colab already has torch, numpy, scipy, sklearn, matplotlib.
!pip install -q pyyaml h5py

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU ONLY — stop and switch the runtime to GPU')

## 2. Get the data in — use Drive, not the upload button

`X.npy` is now **502 MB** (128,400 × 2 × 512 float32); the three files total
**507 MB**. `files.upload()` pushes that through the browser tab and routinely
stalls or dies at this size, so Drive is the primary path here rather than the
alternative it was in the older notebooks.

**On your machine, once:** upload `data/processed/X.npy`, `y.npy` and
`snr_labels.npy` into a `sedic/` folder in your Google Drive (drive.google.com,
drag them in). They persist, so later sessions skip straight to the copy below.

Do NOT rebuild the dataset here — `data/raw/` holds a 21 GB RadioML HDF5 that
is not in the repo and would have to be uploaded too. Build locally, upload the
three arrays.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p data/processed
!cp /content/drive/MyDrive/sedic/X.npy          data/processed/
!cp /content/drive/MyDrive/sedic/y.npy          data/processed/
!cp /content/drive/MyDrive/sedic/snr_labels.npy data/processed/
!ls -la data/processed/

In [ ]:
# Fallback ONLY if Drive is not an option. Expect this to be slow and to fail
# sometimes at ~500 MB; re-running the Drive cell above is the better fix.
#
# import shutil, os
# from google.colab import files
# os.makedirs('data/processed', exist_ok=True)
# for name in files.upload():
#     shutil.move(name, f'data/processed/{name}')

## 3. Verify the data BEFORE spending GPU time

This is the cell that would have caught the old dataset. It checks the shape,
the multi-hot label format, **and that the SNR bins actually match the config** —
the specific thing that was wrong before. Seconds to run.

**If any assert fires, stop.** Do not run the training cells.

In [ ]:
import numpy as np
from src.config import CFG, CLASSES

X = np.load('data/processed/X.npy', mmap_mode='r')
y = np.load('data/processed/y.npy')
snr = np.load('data/processed/snr_labels.npy')

print('X  ', X.shape, X.dtype)
print('y  ', y.shape, y.dtype)
print('snr', snr.shape)

assert X.ndim == 3 and X.shape[1] == 2, f'expected (N, 2, L), got {X.shape}'
assert y.ndim == 2 and y.shape[1] == len(CLASSES), (
    f'y must be multi-hot (N, {len(CLASSES)}) — got {y.shape}. An (N,) array is '
    'the OLD single-label format; rebuild with python -m src.data.build_dataset.')
assert len(X) == len(y) == len(snr), 'X / y / snr lengths disagree'

actual = sorted(int(v) for v in set(snr.tolist()))
expected = sorted(CFG['snr_bins_db'])
print('\nSNR bins  config:', expected)
print('SNR bins  actual:', actual)
assert actual == expected, (
    'SNR bins do NOT match the config. This is exactly the defect in the old '
    'data/processed (it had only [0, 10]). Rebuild locally and re-upload.')

n = y.sum(axis=1)
ni = CLASSES.index('NOISE_FLOOR')
assert (n == 0).sum() == 0, 'some windows carry no label at all'
assert ((y[:, ni] > 0.5) & (n > 1)).sum() == 0, (
    'NOISE_FLOOR co-occurs with another class. The dataset guarantees it never '
    'does, and the display-layer noise gate relies on that.')

print('\nper SNR bin :', {b: int((snr == b).sum()) for b in expected})
print('standalone  :', int((n == 1).sum()))
print('composite   :', int((n > 1).sum()))
print('\nOK — data matches the config.')

In [ ]:
# Code sanity check. Fast, and catches a bad clone or a missing dependency
# before the long run rather than after it.
!python -m pytest -q

## 4. Train the ensemble

This is the run that matters. Five seeds, sigmoid outputs averaged — a single
run cannot answer "is it above 80%" on its own, because seed-to-seed spread on
this project has been measured at up to **10.8 points on JAMMING recall alone**
(0.728–0.836 across five identical runs).

Writes `results/ensemble_0.pt` … `ensemble_4.pt`.

Long cell. Colab disconnects idle tabs — leave it open and visible.

In [ ]:
!python scripts/train_ensemble.py --models 5

In [ ]:
# Single-model baseline -> results/best_model.pt.
# Not used by the ensemble scorecard, but the app's Model dropdown offers it
# and the static build exports it, so it is worth the few extra minutes.
!python -m src.train

In [ ]:
!python -m src.evaluate

### Optional: how much of this is just seed noise?

`measure_variance.py` retrains the model five more times purely to report the
spread. It writes **no checkpoints** and nothing downstream depends on it — it
answers "is this change real or luck?", which matters when tuning and not when
you just want a trained ensemble.

It roughly **doubles** the GPU time of this notebook. Skip it unless you are
comparing against a previous configuration.

In [ ]:
# !python scripts/measure_variance.py --runs 5

## 5. Did it pass?

Only **LFM_RADAR, FHSS and JAMMING** are judged (`judged_classes` in the
config); the civilian classes and NOISE_FLOOR are not scored by the benchmark.
The bar is 80% recall.

These numbers use the thresholds currently in `configs/default.yaml`, which
were calibrated against the **previous** checkpoints. Expect them to be a
little off here — calibration happens after the download, and is what the
final numbers should be read from.

In [ ]:
import json, pathlib

for name, path in [('Single model', 'evals/scorecard.json'),
                    ('Ensemble', 'evals/ensemble_scorecard.json')]:
    p = pathlib.Path(path)
    if not p.is_file():
        print(f'--- {name}: {path} not written ---\n')
        continue
    sc = json.loads(p.read_text())
    print(f'--- {name} ({path}) ---')
    bench = sc.get('benchmark', {})
    for cls, r in bench.get('judged_classes', {}).items():
        print(f"  {cls:<12} recall={r['recall']:.4f}  "
              f"{'PASS' if r['passed'] else 'FAIL'}")
    print(f"  OVERALL: {'PASS' if bench.get('passed') else 'FAIL'}\n")

In [ ]:
from IPython.display import Image, display
display(Image('evals/confusion_matrix.png'))
display(Image('evals/accuracy_vs_snr.png'))

## 6. Get the checkpoints out — DO NOT SKIP

Colab deletes everything when the runtime ends. Copying to Drive is safer than
the browser download, which fails silently on multi-file grabs.

In [ ]:
!mkdir -p /content/drive/MyDrive/sedic/results_new
!cp results/ensemble_*.pt  /content/drive/MyDrive/sedic/results_new/
!cp results/best_model.pt  /content/drive/MyDrive/sedic/results_new/
!cp -r evals              /content/drive/MyDrive/sedic/results_new/
!ls -la /content/drive/MyDrive/sedic/results_new/

In [ ]:
# Browser download instead of / as well as Drive.
#
# from google.colab import files
# for i in range(5):
#     files.download(f'results/ensemble_{i}.pt')
# files.download('results/best_model.pt')

## 7. Finish locally — calibration is NOT run here

`calibrate_thresholds.py` selects thresholds on the **validation** split and
reports on test. It has to run against the same `data/processed` and the same
checkpoints you will actually ship, so it belongs on the machine holding both —
not in a runtime that is about to be deleted.

On your machine, with the new `ensemble_*.pt` and `best_model.pt` copied into
`results/`:

```bash
python scripts/calibrate_thresholds.py --ensemble --n-models 5
```

Then put the printed values into `configs/default.yaml` under
`multilabel_thresholds_per_class`, and confirm:

```bash
python -m src.evaluate --ensemble
```

**The thresholds are load-bearing.** Last calibration moved LFM_RADAR from
79.61% (FAIL) to 82.25% (PASS) on the same checkpoints — the config says so in
its own comments. New checkpoints without new thresholds is not a valid result.

### If you are also republishing the static site

```bash
python scripts/export_onnx.py   # .pt  -> results/onnx/
python web/build.py             # -> web/models/, web/data/
```

and copy the new thresholds into `THRESHOLDS` in `web/analysis.js` — it is the
one config value the static build duplicates, and `web/test/analysis_check.mjs`
will fail if the two drift apart.